<a href="https://colab.research.google.com/github/buiswrld/ecg-leads/blob/main/MSNS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1: Download and Unzip the PTBXL dataset

### Load python packages

In [28]:
import pandas as pd
import numpy as np
import wfdb
import ast
import gdown
import zipfile
import os
from pathlib import Path
import torch
from torch.utils.data import Dataset
from sklearn.preprocessing import MultiLabelBinarizer

### Download and unzip Google Drive Files

In [ ]:
# Configuration
file_id = "1f4sk1pCJ6SKK8M-TRpEhEiVqvLTHV80p"
output_zip = "download_file.zip"
extract_folder = "unzipped_files"
# Download the Google Drive file
url = f"https://drive.google.com/uc?id={file_id}"
gdown.download(url, output_zip, quiet = False)
# Unzip the zip file
os.makedirs(extract_folder, exist_ok=True)
with zipfile.ZipFile(output_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

### Access unzipped datasets

In [7]:
# Configuration
# Replace path with your own path
path = '.\\unzipped_files\\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1\\'
sampling_rate=100

## Step 2: Load and Aggregate Metadata

In [5]:
# Functions
def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

In [8]:
# Load and convert annotation data
Y = pd.read_csv(path+'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

In [9]:
# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

In [10]:
# Apply diagnostic superclass
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)

In [11]:
# Save the processed metadata to a new CSV file
Y.to_csv("processed_ptbxl_metadata.csv")

## Step 3: Load raw ECG data

In [ ]:
# Functions
def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

In [6]:
# Load raw signal data: sampling_rate=100 or 500
X = load_raw_data(Y, sampling_rate, path)

In [7]:
# Save X numpy.ndarray object
np.save("X_numpy_ndarray.npy", X)

## Step 4: Define PTBXLDataset class

In [42]:
class PTBXLDataset(Dataset):
    def __init__(self, csv_path, data_root, split="train", leads=None, mlb=None):
        """
        csv_path: path to 'processed_ptbxl_metadata.csv'
        data_root: folder directory containing 'X_numpy_ndarray.npy'
        split: "train", "val", or "test"
        leads: optional list of lead names, e.g. ["I", "II", "V1"]
        """
        self.split = split
        
        # 1. Load the pre-processed metadata CSV created in Step 2
        df = pd.read_csv(csv_path, index_col='ecg_id')
        
        # Convert string representations of lists back to actual Python lists
        df['diagnostic_superclass'] = df['diagnostic_superclass'].apply(lambda x: ast.literal_eval(x))

        # 2. Filter the index using the recommended stratification folds
        if split == "train":
            mask = (df.strat_fold <= 8).values
        elif split == "val":
            mask = (df.strat_fold == 9).values
        elif split == "test":
            mask = (df.strat_fold == 10).values
        else:
            raise ValueError("Split must be 'train', 'val', or 'test'")
            
        # Keep only the rows for this specific split
        self.df = df[mask].reset_index(drop=True)
        
        # 3. Load the pre-saved numpy matrix from Step 3 and slice it with the exact same mask
        full_X = np.load(os.path.join(data_root, "X_numpy_ndarray.npy"))
        self.X = full_X[mask]
        
        # 4. Handle optional lead selection filtering
        self.lead_map = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
        self.lead_indices = [self.lead_map.index(l) for l in leads] if leads else None

        # 5. Set up the Label Encoder to convert text to numeric vectors
        # If classes are provided (for val/test), use them. Otherwise, fit to the training labels.
        if mlb is None:
            # If no mlb is provided (Training set), create a new one and fit it
            self.mlb = MultiLabelBinarizer()
            self.encoded_labels = self.mlb.fit_transform(self.df['diagnostic_superclass'])
        else:
            # If a pre-fitted mlb is passed (Val/Test sets), reuse it exactly
            self.mlb = mlb
            self.encoded_labels = self.mlb.transform(self.df['diagnostic_superclass'])

    def __len__(self):
        # Return the exact number of rows in this data split
        return len(self.df)

    def __getitem__(self, index):
        # 1. Grab the matrix slice from memory
        ecg_signal = self.X[index]  # Shape: (1000, 12)
        
        # 2. Filter out specific leads if requested
        if self.lead_indices is not None:
            ecg_signal = ecg_signal[:, self.lead_indices]
            
        # 3. Convert to tensor and flip shape to match PyTorch expectations: (channels, time_steps)
        ecg = torch.tensor(ecg_signal, dtype=torch.float32).transpose(0, 1)
        
        # 4. Extract the numeric binary tensor label instead of a text list
        label = torch.tensor(self.encoded_labels[index], dtype=torch.float32)

        return {
            "ecg": ecg,
            "label": label
        }

In [ ]:
# Generate datasets for each split
# 1. Initialize your training dataset
train_dataset = PTBXLDataset(csv_path="processed_ptbxl_metadata.csv", data_root=".", split="train")

# 2. validation and test datesets
val_dataset = PTBXLDataset(csv_path="processed_ptbxl_metadata.csv", data_root=".", split="val", mlb=train_dataset.mlb)
test_dataset = PTBXLDataset(csv_path="processed_ptbxl_metadata.csv", data_root=".", split="test", mlb=train_dataset.mlb)